# Import

In [1]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
 
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
import shap
 
print("=" * 60)
print("ML EXPERIMENT — XGBoost 30-Day Readmission Classifier")
print("=" * 60)


StatementMeta(, 1f230dc0-6974-44b9-9ee9-cd791f1a0bb0, 3, Finished, Available, Finished, False)

ML EXPERIMENT — XGBoost 30-Day Readmission Classifier


##  Configure MLflow

In [2]:
EXPERIMENT_NAME = "HospitalReadmission_30Day"
mlflow.set_experiment(EXPERIMENT_NAME)

StatementMeta(, 1f230dc0-6974-44b9-9ee9-cd791f1a0bb0, 4, Finished, Available, Finished, False)

2026/04/27 10:23:17 INFO mlflow.tracking.fluent: Experiment with name 'HospitalReadmission_30Day' does not exist. Creating a new experiment.


<Experiment: artifact_location='sds://onelakecentralindia.pbidedicated.windows.net/d7ddf886-b4b2-42a7-bbe6-4f256e6573a2/a0887097-a342-4742-8474-98dd8c25c70a', creation_time=1777285399123, experiment_id='a0887097-a342-4742-8474-98dd8c25c70a', last_update_time=1777285399123, lifecycle_stage='active', name='HospitalReadmission_30Day', tags={}>

## Load Features

In [3]:
FEATURE_COLS = [
    "time_in_hospital","num_lab_procedures","num_procedures",
    "num_medications","number_diagnoses",
    "number_inpatient","number_emergency","number_outpatient",
    "prior_visits_total","is_high_prior_use",
    "has_prior_inpatient","has_prior_emergency",
    "is_long_stay","is_polypharmacy","is_complex_patient",
    "age_ord",
    "total_med_changes","total_meds_taken",
    "insulin_changed","insulin_increased",
    "A1C_tested","A1C_high","A1C_normal","glucose_tested",
    "is_diabetes_primary","has_circulatory_dx",
    "diag1_cat_idx","diag2_cat_idx","diag3_cat_idx",
    "gender_idx","race_idx",
]
TARGET = "readmitted_30d"
 
df_spark   = spark.read.format("delta").table("silver_features")
available  = [f for f in FEATURE_COLS if f in df_spark.columns]
print(f"Features available: {len(available)}")
 
df_pd = df_spark.select(["encounter_id","patient_nbr", TARGET] + available) \
               .toPandas()
print(f"Loaded {len(df_pd):,} patient records")
 
X = df_pd[available].fillna(0)
y = df_pd[TARGET]

StatementMeta(, 1f230dc0-6974-44b9-9ee9-cd791f1a0bb0, 5, Finished, Available, Finished, False)

Features available: 31
Loaded 71,518 patient records


## Class Imbalance Analysis

In [4]:
# The dataset has 4.5% positive rate (readmitted within 30 days).
# XGBoost's scale_pos_weight parameter compensates for this.
# Formula: (count negative) / (count positive)
pos_count  = y.sum()
neg_count  = len(y) - pos_count
scale_pw   = neg_count / pos_count
pos_rate   = y.mean()
 
print(f"\nClass distribution:")
print(f"  Positive (readmitted): {pos_count:,} ({pos_rate:.1%})")
print(f"  Negative (not):        {neg_count:,} ({1-pos_rate:.1%})")
print(f"  scale_pos_weight:      {scale_pw:.1f}")
 
# XGBoost Parameters ────────────────────────────
XGB_PARAMS = dict(
    n_estimators       = 500,
    learning_rate      = 0.05,
    max_depth          = 6,
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    min_child_weight   = 5,
    gamma              = 0.1,
    reg_alpha          = 0.1,
    reg_lambda         = 1.0,
    scale_pos_weight   = scale_pw,
    eval_metric        = "auc",
    use_label_encoder  = False,
    random_state       = 42,
    n_jobs             = -1,
    verbosity          = 0,
)

StatementMeta(, 1f230dc0-6974-44b9-9ee9-cd791f1a0bb0, 6, Finished, Available, Finished, False)


Class distribution:
  Positive (readmitted): 3,224 (4.5%)
  Negative (not):        68,294 (95.5%)
  scale_pos_weight:      21.2


## Cross-Validation

In [5]:
# Stratified k-fold ensures each fold has the same 4.5% positive rate.
# This gives a robust, unbiased AUC estimate before the final model fit.
print("\nRunning 5-fold stratified cross-validation...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_aucs = cross_val_score(
    XGBClassifier(**XGB_PARAMS), X, y,
    cv=cv, scoring="roc_auc", n_jobs=-1
)
print(f"CV AUC-ROC: {cv_aucs.mean():.4f} ± {cv_aucs.std():.4f}")

StatementMeta(, 1f230dc0-6974-44b9-9ee9-cd791f1a0bb0, 7, Finished, Available, Finished, False)


Running 5-fold stratified cross-validation...
CV AUC-ROC: 0.6433 ± 0.0074


##  Final Model + MLflow Run

In [6]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
 
with mlflow.start_run(run_name="XGBoost_Readmit30d_v1"):
 
    model = XGBClassifier(**XGB_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_te, y_te)],
        early_stopping_rounds=50,
        verbose=False
    )
 
    # Predictions
    y_prob = model.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= 0.35).astype(int)  # 35% threshold = HIGH tier
 
    # Metrics
    auc = roc_auc_score(y_te, y_prob)
    ap  = average_precision_score(y_te, y_prob)
 
    # SHAP explainability (sample 500 rows for speed)
    explainer   = shap.TreeExplainer(model)
    shap_vals   = explainer.shap_values(X_te.iloc[:500])
    fi_df = pd.DataFrame({
        "feature":    available,
        "mean_shap":  np.abs(shap_vals).mean(axis=0)
    }).sort_values("mean_shap", ascending=False)
 
    # Log to MLflow
    mlflow.log_param("n_features",        len(available))
    mlflow.log_param("train_size",        len(X_tr))
    mlflow.log_param("test_size",         len(X_te))
    mlflow.log_param("pos_rate",          round(pos_rate, 4))
    mlflow.log_param("scale_pos_weight",  round(scale_pw, 2))
    mlflow.log_params({f"xgb_{k}": v for k, v in XGB_PARAMS.items()})
    mlflow.log_metric("AUC_ROC",          round(auc, 4))
    mlflow.log_metric("Avg_Precision",    round(ap, 4))
    mlflow.log_metric("CV_AUC_mean",      round(cv_aucs.mean(), 4))
    mlflow.log_metric("CV_AUC_std",       round(cv_aucs.std(), 4))
    mlflow.log_metric("best_iteration",   model.best_iteration)
 
    # Save feature importance to MLflow artifact
    fi_path = "/tmp/feature_importance_readmit.csv"
    fi_df.to_csv(fi_path, index=False)
    mlflow.log_artifact(fi_path, artifact_path="feature_importance")
 
    # Register model
    mlflow.sklearn.log_model(
        model,
        artifact_path="xgb_readmission",
        registered_model_name="HospitalReadmission30d"
    )
 
    print(f"\n{'='*50}")
    print(f"AUC-ROC:       {auc:.4f}")
    print(f"Avg Precision: {ap:.4f}")
    print(f"Best iter:     {model.best_iteration}")
    print(f"\nTop 10 features by SHAP importance:")
    print(fi_df.head(10).to_string(index=False))

StatementMeta(, 1f230dc0-6974-44b9-9ee9-cd791f1a0bb0, 10, Finished, Available, Finished, False)

`early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
[10:38:21] WARNING: /croot/xgboost-split_1713972711803/work/cpp_src/src/c_api/c_api.cc:1240: Saving into deprecated binary model format, please consider using `json` or `ubj`. Model format will default to JSON in XGBoost 2.2 if not specified.
Setuptools is replacing distutils.



AUC-ROC:       0.6647
Avg Precision: 0.0942
Best iter:     32

Top 10 features by SHAP importance:
            feature  mean_shap
   number_inpatient   0.167818
            age_ord   0.155919
 num_lab_procedures   0.104633
 prior_visits_total   0.090865
   number_diagnoses   0.076729
   time_in_hospital   0.069683
    num_medications   0.049272
has_prior_inpatient   0.043958
           race_idx   0.030464
      diag1_cat_idx   0.027084


## Performance Interpretation

In [8]:
print(f"\n{'='*50}")
print("MODEL PERFORMANCE GUIDE")
print("="*50)
print(f"AUC-ROC: {auc:.4f}")
if auc >= 0.75:
    print("  → EXCELLENT for clinical readmission prediction")
elif auc >= 0.70:
    print("  → GOOD — competitive with published literature")
elif auc >= 0.65:
    print("  → ACCEPTABLE — better than no model, room to improve")
else:
    print("  → BELOW TARGET — review feature engineering")
 
print(f"\nProceed to → 05_risk_scoring.py")
 

StatementMeta(, 1f230dc0-6974-44b9-9ee9-cd791f1a0bb0, 13, Finished, Available, Finished, False)


MODEL PERFORMANCE GUIDE
AUC-ROC: 0.6647
  → ACCEPTABLE — better than no model, room to improve

Proceed to → 05_risk_scoring.py
